In [2]:
!ls /srv/data/lt2326-h25/a2

hi_hdtb-ud-dev.conllu  hi_hdtb-ud-test.conllu  hi_hdtb-ud-train.conllu


### Part 1 Transformers for Sequence Classification
https://huggingface.co/docs/transformers/tasks/token_classification

#### reading conllu file
https://pypi.org/project/conllu/

In [5]:
!pip install conllu

Defaulting to user installation because normal site-packages is not writeable


In [7]:
!pip show conllu

Name: conllu
Version: 6.0.0
Summary: CoNLL-U Parser parses a CoNLL-U formatted string into a nested python dictionary
Home-page: 
Author: 
Author-email: Emil Stenström <emil@emilstenstrom.se>
License: The MIT License (MIT)
        
        Copyright (c) 2016 Emil Stenström
        
        Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:
        
        The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.
        
        THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO TH

In [4]:
import sys
print(sys.executable)

/usr/bin/python3.12


In [5]:
!{sys.executable} -m pip install conllu

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [10]:
import sys
import os
import numpy as np

In [11]:
def read_hindi_data(inputfilename):
    with open(inputfilename, "r") as inputfile:
        sentences = []
        collection_words = []
        collection_labels = []
        for line in inputfile:
            if line[0] == '#':
                continue
            columns = line.split()
            #print(words)
            if columns == []:
                sentences.append((''.join(collection_words), collection_labels))
                collection_words = []
                collection_labels = []
                continue
            collection_words.append(columns[1])
            collection_labels += [1] + ([0] * (len(columns[1]) - 1))
            
    return sentences

In [12]:
train_sentences = read_hindi_data('/srv/data/lt2326-h25/a2/hi_hdtb-ud-train.conllu')

In [20]:
train_sentences[0]

('यहएशियाकीसबसेबड़ीमस्जिदोंमेंसेएकहै।',
 [1,
  0,
  1,
  0,
  0,
  0,
  0,
  1,
  0,
  1,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  1,
  0,
  1,
  0,
  1,
  0,
  1])

In [13]:
def index_chars(sentences):
    megasentence = ''.join(sentences)
    char_list = set()
    for c in megasentence:
        char_list.add(c)
    char_list = list(char_list)
    return char_list, {char_list[x]:x for x in range(len(char_list))}

In [14]:
int_index, char_index = index_chars([x[0] for x in train_sentences])

In [15]:
int_index

['.',
 'न',
 '6',
 'ऱ',
 'अ',
 '(',
 '६',
 'च',
 '७',
 '०',
 'ऋ',
 '7',
 '४',
 '्',
 'ं',
 '।',
 'ू',
 ')',
 'ॉ',
 'ह',
 'ो',
 'उ',
 'ड',
 ':',
 'ृ',
 'श',
 '4',
 'ल',
 'औ',
 'द',
 'ॅ',
 'त',
 'ौ',
 '\u200d',
 'ढ',
 'N',
 'ञ',
 'ग',
 'व',
 'क',
 '5',
 'र',
 'आ',
 'I',
 '3',
 'J',
 'ए',
 'ः',
 'O',
 'य',
 'ी',
 '8',
 'भ',
 'ख',
 'ओ',
 'ँ',
 '़',
 '८',
 'झ',
 'ऑ',
 '९',
 'फ',
 'ा',
 'ठ',
 '!',
 '/',
 'थ',
 'ण',
 'ब',
 'म',
 'स',
 'ज',
 '1',
 'ि',
 'ु',
 'े',
 'ऊ',
 'इ',
 '-',
 'ऐ',
 'ष',
 '"',
 'ट',
 '१',
 'घ',
 '?',
 ',',
 '५',
 '२',
 'ै',
 '2',
 'प',
 '३',
 "'",
 'ई',
 '0',
 'ध',
 'छ',
 '9']

In [24]:
char_index

{'.': 0,
 'न': 1,
 '6': 2,
 'ऱ': 3,
 'अ': 4,
 '(': 5,
 '६': 6,
 'च': 7,
 '७': 8,
 '०': 9,
 'ऋ': 10,
 '7': 11,
 '४': 12,
 '्': 13,
 'ं': 14,
 '।': 15,
 'ू': 16,
 ')': 17,
 'ॉ': 18,
 'ह': 19,
 'ो': 20,
 'उ': 21,
 'ड': 22,
 ':': 23,
 'ृ': 24,
 'श': 25,
 '4': 26,
 'ल': 27,
 'औ': 28,
 'द': 29,
 'ॅ': 30,
 'त': 31,
 'ौ': 32,
 '\u200d': 33,
 'ढ': 34,
 'N': 35,
 'ञ': 36,
 'ग': 37,
 'व': 38,
 'क': 39,
 '5': 40,
 'र': 41,
 'आ': 42,
 'I': 43,
 '3': 44,
 'J': 45,
 'ए': 46,
 'ः': 47,
 'O': 48,
 'य': 49,
 'ी': 50,
 '8': 51,
 'भ': 52,
 'ख': 53,
 'ओ': 54,
 'ँ': 55,
 '़': 56,
 '८': 57,
 'झ': 58,
 'ऑ': 59,
 '९': 60,
 'फ': 61,
 'ा': 62,
 'ठ': 63,
 '!': 64,
 '/': 65,
 'थ': 66,
 'ण': 67,
 'ब': 68,
 'म': 69,
 'स': 70,
 'ज': 71,
 '1': 72,
 'ि': 73,
 'ु': 74,
 'े': 75,
 'ऊ': 76,
 'इ': 77,
 '-': 78,
 'ऐ': 79,
 'ष': 80,
 '"': 81,
 'ट': 82,
 '१': 83,
 'घ': 84,
 '?': 85,
 ',': 86,
 '५': 87,
 '२': 88,
 'ै': 89,
 '2': 90,
 'प': 91,
 '३': 92,
 "'": 93,
 'ई': 94,
 '0': 95,
 'ध': 96,
 'छ': 97,
 '9': 98}

In [18]:
def convert_sentence(sentence, index):
    return [index[x] for x in sentence]

In [19]:
train_converted = [(convert_sentence(x[0], char_index), x[1]) for x in train_sentences]

In [26]:
train_converted[0]

([49,
  19,
  46,
  25,
  73,
  49,
  62,
  39,
  50,
  70,
  68,
  70,
  75,
  68,
  22,
  56,
  50,
  69,
  70,
  13,
  71,
  73,
  29,
  20,
  14,
  69,
  75,
  14,
  70,
  75,
  46,
  39,
  19,
  89,
  15],
 [1,
  0,
  1,
  0,
  0,
  0,
  0,
  1,
  0,
  1,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  1,
  0,
  1,
  0,
  1,
  0,
  1])

In [27]:
train_X, train_y = zip(*train_converted)

In [43]:
train_X[0]

[49,
 19,
 46,
 25,
 73,
 49,
 62,
 39,
 50,
 70,
 68,
 70,
 75,
 68,
 22,
 56,
 50,
 69,
 70,
 13,
 71,
 73,
 29,
 20,
 14,
 69,
 75,
 14,
 70,
 75,
 46,
 39,
 19,
 89,
 15]

In [44]:
train_y[0]

[1,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 1]

In [31]:
lengths = [len(x) for x in train_X]
max_length = max(lengths)

In [32]:
def pad_lengths(sentences, max_length):
    return [x + ([-1] * (max_length - len(x))) for x in sentences]

In [33]:
train_padded_X = pad_lengths(train_X, max_length)

In [40]:
import torch

In [37]:
train_X_tensor = torch.LongTensor(train_padded_X)

In [38]:
train_X_tensor.size()

torch.Size([13306, 429])

In [39]:
train_X_tensor

tensor([[49, 19, 46,  ..., -1, -1, -1],
        [77, 70, 75,  ..., -1, -1, -1],
        [77, 70, 39,  ..., -1, -1, -1],
        ...,
        [21,  1, 13,  ..., -1, -1, -1],
        [77,  1, 29,  ..., -1, -1, -1],
        [21,  1, 13,  ..., -1, -1, -1]])

In [41]:
train_y_tensor = torch.LongTensor(train_y)

ValueError: expected sequence of length 35 at dim 1 (got 25)